# EHR-M-GAN (Neo M3GAN V2) Training on Kaggle

This notebook is prepared for training the improved **neo_m3gan_v2** model. It includes 3-layer MLP Mapping Networks in G, Minibatch Standard Deviation in D, and a corrected evaluation pipeline to prevent continuous data mode collapse.

### Before you start:
1. **GPU Required**: Go to **Settings** (right sidebar) -> **Accelerator** -> Select **GPU T4 x2** (or P100).
2. **Internet Required**: Go to **Settings** -> **Internet** -> Select **On**.
3. **Add Data**: Click **+ Add Data** -> Search for your dataset **`mimic3`** and add it.

---

In [ ]:
# 1. Create directory structure
import os
os.makedirs('Data/mimic', exist_ok=True)
os.makedirs('Output/checkpoint', exist_ok=True)
os.makedirs('logs', exist_ok=True)

# 2. Link Kaggle Data to local Data directory
dataset_root = '/kaggle/input/datasets/lmnhthng/mimic3'

for filename in ['vital_sign_24hrs.pkl', 'med_interv_24hrs.pkl']:
    src = os.path.join(dataset_root, filename)
    dst = os.path.join('Data/mimic', filename)
    if os.path.exists(src):
        if os.path.exists(dst): os.remove(dst)
        os.symlink(src, dst)
        print(f"Linked {filename}")
    else:
        print(f"WARNING: {filename} not found in {dataset_root}")

In [ ]:
%%writefile ultils.py
import torch
import torch.nn.functional as F
import numpy as np

def nt_xent_loss(out_1, out_2, temperature=1.0):
    batch_size = out_1.shape[0]
    out_1 = F.normalize(out_1, p=2, dim=-1)
    out_2 = F.normalize(out_2, p=2, dim=-1)
    out = torch.cat([out_1, out_2], dim=0)
    cov = torch.mm(out, out.t().contiguous())
    sim = torch.exp(cov / temperature)
    mask = ~torch.eye(2 * batch_size, device=out.device).bool()
    negatives = sim.masked_select(mask).view(2 * batch_size, -1)
    positives = torch.exp(torch.sum(out_1 * out_2, dim=-1) / temperature)
    positives = torch.cat([positives, positives], dim=0)
    loss = -torch.log(positives / (negatives.sum(dim=-1) + 1e-8) + 1e-8)
    return loss.mean()

def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1))

def feature_matching_loss(fake_features, real_features):
    mean_fake = torch.mean(fake_features, dim=0)
    mean_real = torch.mean(real_features, dim=0)
    std_fake = torch.sqrt(torch.var(fake_features, dim=0) + 1e-6)
    std_real = torch.sqrt(torch.var(real_features, dim=0) + 1e-6)
    loss_mean = torch.mean(torch.abs(mean_fake - mean_real))
    loss_std = torch.mean(torch.abs(std_fake - std_real))
    return loss_mean + loss_std

def renormlizer(data, max_val, min_val):
    data = data * (max_val + 1e-8)
    data = data + min_val
    return data

def np_rounding(prob):
    y = np.round(prob)
    return y


In [ ]:
%%writefile networks.py
import torch
import torch.nn as nn
from torch.nn.utils import spectral_norm

class TemporalSelfAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalSelfAttention, self).__init__()
        self.query = nn.Linear(hidden_dim, hidden_dim // 8)
        self.key = nn.Linear(hidden_dim, hidden_dim // 8)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x shape: (batch_size, time_steps, hidden_dim)
        batch_size, time_steps, hidden_dim = x.size()
        
        proj_query = self.query(x) # B x T x C (where C = hidden_dim // 8)
        proj_key = self.key(x).permute(0, 2, 1) # B x C x T
        
        energy = torch.bmm(proj_query, proj_key) # B x T x T
        attention = torch.softmax(energy, dim=-1)
        
        proj_value = self.value(x) # B x T x H
        out = torch.bmm(attention, proj_value) # B x T x H
        
        out = self.gamma * out + x
        return out


class VAE_Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers=3):
        super(VAE_Encoder, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x_t, hidden_state):
        # x_t shape: [batch_size, 1, input_dim]
        out, new_hidden = self.lstm(x_t, hidden_state)
        out_squeeze = out.squeeze(1)

        mu = self.fc_mu(out_squeeze)
        logvar = self.fc_logvar(out_squeeze)

        # Defensive clamps to prevent numerical instability
        mu = torch.clamp(mu, min=-20.0, max=20.0)
        logvar = torch.clamp(logvar, min=-20.0, max=20.0)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z, mu, logvar, new_hidden


class VAE_Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, num_layers=3):
        super(VAE_Decoder, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(latent_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z_t, hidden_state):
        # z_t shape: [batch_size, 1, latent_dim]
        out, new_hidden = self.lstm(z_t, hidden_state)
        logits = self.fc_out(out.squeeze(1))
        reconstruction = torch.sigmoid(logits)
        return reconstruction, logits, new_hidden


class AutoregressiveVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, enc_layers, dec_layers, time_steps):
        super(AutoregressiveVAE, self).__init__()
        self.time_steps = time_steps
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.encoder = VAE_Encoder(input_dim * 2, hidden_dim, latent_dim, enc_layers)
        self.decoder = VAE_Decoder(latent_dim, hidden_dim, input_dim, dec_layers)

    def forward(self, x):
        batch_size = x.size(0)
        device = x.device

        # Initialize hidden states
        enc_hidden = (torch.zeros(self.encoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.encoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))

        c_prev = torch.zeros(batch_size, self.input_dim, device=device)

        rec_list, logits_list, mu_list, logvar_list, z_list = [], [], [], [], []

        for t in range(self.time_steps):
            x_t = x[:, t, :]
            c_sigmoid = torch.sigmoid(c_prev)
            x_hat = x_t - c_sigmoid

            enc_in = torch.cat([x_t, x_hat], dim=1).unsqueeze(1)
            z_t, mu_t, logvar_t, enc_hidden = self.encoder(enc_in, enc_hidden)

            z_in = z_t.unsqueeze(1)
            rec_t, logits_t, dec_hidden = self.decoder(z_in, dec_hidden)

            c_prev = logits_t

            rec_list.append(rec_t.unsqueeze(1))
            logits_list.append(logits_t.unsqueeze(1))
            mu_list.append(mu_t.unsqueeze(1))
            logvar_list.append(logvar_t.unsqueeze(1))
            z_list.append(z_t.unsqueeze(1))

        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1), \
            torch.cat(mu_list, dim=1), torch.cat(logvar_list, dim=1), torch.cat(z_list, dim=1)

    def reconstruct_decoder(self, z_seq):
        batch_size = z_seq.size(0)
        device = z_seq.device
        dec_hidden = (torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device),
                      torch.zeros(self.decoder.lstm.num_layers, batch_size, self.hidden_dim, device=device))

        rec_list, logits_list = [], []
        for t in range(self.time_steps):
            z_t = z_seq[:, t, :].unsqueeze(1)
            rec_t, logits_t, dec_hidden = self.decoder(z_t, dec_hidden)
            rec_list.append(rec_t.unsqueeze(1))
            logits_list.append(logits_t.unsqueeze(1))

        return torch.cat(rec_list, dim=1), torch.cat(logits_list, dim=1)


class SequenceDiscriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim, time_steps, num_layers=3):
        super(SequenceDiscriminator, self).__init__()
        dropout_rate = 0.2 if num_layers > 1 else 0.0
        # Tăng thêm 1 chiều đầu vào (input_dim + 1) để chứa kênh thống kê Minibatch StdDev
        self.lstm = nn.LSTM(input_dim + 1, hidden_dim, num_layers, batch_first=True, dropout=dropout_rate)
        # Add Self Attention for advanced global pooling
        self.attn = TemporalSelfAttention(hidden_dim)
        # Use spectral normalization to enforce Lipschitz continuity for WGAN-GP
        self.fc = spectral_norm(nn.Linear(hidden_dim * time_steps, 1))

    def forward(self, x):
        # x shape: [batch_size, time_steps, input_dim]
        batch_size, time_steps, channels = x.size()
        
        # Feature-wise Standard Deviation (Per-Sample Channel StdDev)
        # Computes variance across features/channels for each sample independently.
        # This completely avoids cross-sample gradient leakage during WGAN-GP double-backpropagation
        # and remains highly numerically stable since a single patient's features are never all constant.
        var = torch.var(x, dim=-1, keepdim=True, unbiased=False)
        std_mean = torch.sqrt(var + 1e-4)
        
        # Concatenate standard deviation to input features
        x_concat = torch.cat([x, std_mean], dim=-1) # shape: [batch_size, time_steps, input_dim + 1]

        out, _ = self.lstm(x_concat)
        out = self.attn(out)
        out_flat = torch.flatten(out, start_dim=1)
        logits = self.fc(out_flat).squeeze(-1)  # Output an unbounded critic score for WGAN
        return logits, out


class BilateralLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BilateralLSTMCell, self).__init__()
        self.hidden_dim = hidden_dim
        self.ln = nn.Linear(input_dim + hidden_dim + hidden_dim, 4 * hidden_dim, bias=False)

    def forward(self, x, h_self, c_self, h_coupled):
        combined = torch.cat([x, h_self, h_coupled], dim=1)
        gates = self.ln(combined)
        i_gate, f_gate, o_gate, c_tilde = gates.chunk(4, dim=1)

        i = torch.sigmoid(i_gate)
        f = torch.sigmoid(f_gate)
        o = torch.sigmoid(o_gate)
        c_ = torch.tanh(c_tilde)

        c_next = f * c_self + i * c_
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class BilateralGenerator(nn.Module):
    def __init__(self, noise_dim, hidden_dim, latent_dim, num_layers=3):
        super(BilateralGenerator, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        self.cl = nn.ModuleList([
            BilateralLSTMCell(
                input_dim=noise_dim if i == 0 else hidden_dim,
                hidden_dim=hidden_dim
            ) for i in range(num_layers)
        ])
        self.fc_out = nn.Linear(hidden_dim, latent_dim)

    def forward(self, noise_seq, h_coupled_states):
        batch_size, time_steps, _ = noise_seq.size()
        h_states = [torch.zeros(batch_size, self.hidden_dim, device=noise_seq.device) for _ in range(self.num_layers)]
        c_states = [torch.zeros(batch_size, self.hidden_dim, device=noise_seq.device) for _ in range(self.num_layers)]
        outputs = []

        for t in range(time_steps):
            x_t = noise_seq[:, t, :]
            for i in range(self.num_layers):
                h_cpl = h_coupled_states[i]
                h_states[i], c_states[i] = self.cl[i](x_t, h_states[i], c_states[i], h_cpl)
                x_t = h_states[i]

            out_t = torch.sigmoid(self.fc_out(x_t))
            outputs.append(out_t.unsqueeze(1))

        return torch.cat(outputs, dim=1), h_states


class MappingNetwork(nn.Module):
    def __init__(self, input_dim, output_dim, num_layers=3):
        super(MappingNetwork, self).__init__()
        layers = []
        curr_dim = input_dim
        for _ in range(num_layers):
            layers.append(nn.Linear(curr_dim, output_dim))
            layers.append(nn.LeakyReLU(0.2, inplace=False))
            curr_dim = output_dim
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class JointGenerator(nn.Module):
    def __init__(self, c_noise_dim, d_noise_dim, hidden_dim, c_latent_dim, d_latent_dim, num_layers=3):
        super(JointGenerator, self).__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        
        # Mapping Networks to transform standard Gaussian noise into intermediate latent space W (prevents mode collapse)
        self.c_map = MappingNetwork(c_noise_dim, c_noise_dim, num_layers=3)
        self.d_map = MappingNetwork(d_noise_dim, d_noise_dim, num_layers=3)

        self.c_gen = BilateralGenerator(c_noise_dim, hidden_dim, c_latent_dim, num_layers)
        self.d_gen = BilateralGenerator(d_noise_dim, hidden_dim, d_latent_dim, num_layers)

        self.c_attn = TemporalSelfAttention(hidden_dim)
        self.d_attn = TemporalSelfAttention(hidden_dim)

    def forward(self, noise_c, noise_d):
        """
        Implements the step-by-step bilateral coupling required by EHR-M-GAN,
        enhanced with sequence-level Self-Attention and mapping networks.
        """
        batch_size, time_steps, c_noise_dim = noise_c.size()
        d_noise_dim = noise_d.size(-1)
        device = noise_c.device

        # Pass noise through the mapping networks to disentangle the latent space
        noise_c_mapped = self.c_map(noise_c.view(-1, c_noise_dim)).view(batch_size, time_steps, c_noise_dim)
        noise_d_mapped = self.d_map(noise_d.view(-1, d_noise_dim)).view(batch_size, time_steps, d_noise_dim)

        # Initialize hidden and cell states for both streams
        c_h = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        c_c = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        d_h = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]
        d_c = [torch.zeros(batch_size, self.hidden_dim, device=device) for _ in range(self.num_layers)]

        c_features_list = []
        d_features_list = []

        for t in range(time_steps):
            # 1. Capture current noise inputs
            noise_c_t = noise_c_mapped[:, t, :]
            noise_d_t = noise_d_mapped[:, t, :]

            # 2. Coupled inputs come from the OTHER stream's PREVIOUS hidden state
            c_h_coupled = d_h
            d_h_coupled = c_h

            # 3. Step C Generator Layers
            c_x = noise_c_t
            for i in range(self.num_layers):
                c_h[i], c_c[i] = self.c_gen.cl[i](c_x, c_h[i], c_c[i], c_h_coupled[i])
                c_x = c_h[i]
            
            c_features_list.append(c_x.unsqueeze(1))

            # 4. Step D Generator Layers
            d_x = noise_d_t
            for i in range(self.num_layers):
                d_h[i], d_c[i] = self.d_gen.cl[i](d_x, d_h[i], d_c[i], d_h_coupled[i])
                d_x = d_h[i]
            
            d_features_list.append(d_x.unsqueeze(1))

        # Apply Global Attention across the sequence to fix discrete pattern loss
        c_temporal = torch.cat(c_features_list, dim=1)
        d_temporal = torch.cat(d_features_list, dim=1)

        c_attn_out = self.c_attn(c_temporal)
        d_attn_out = self.d_attn(d_temporal)

        # Co-scale latent bounds using tanh * 3.0 to match pretrained VAE distribution and prevent numerical explosion
        fake_z_c = torch.tanh(self.c_gen.fc_out(c_attn_out)) * 3.0
        fake_z_d = torch.tanh(self.d_gen.fc_out(d_attn_out)) * 3.0

        return fake_z_c, fake_z_d


In [ ]:
%%writefile metrics.py
import numpy as np
from sklearn.metrics.pairwise import rbf_kernel
import warnings
import torch


def discrete_probability_rmse(real_d, fake_d):
    """
    Calculates the Root Mean Square Error (RMSE) between the
    dimension-wise probabilities of the real and synthetic discrete data.
    """
    # Calculate probability of each code occurring across all samples and timesteps
    prob_real = np.mean(real_d, axis=(0, 1))
    prob_fake = np.mean(fake_d, axis=(0, 1))

    rmse = np.sqrt(np.mean((prob_real - prob_fake) ** 2))
    return rmse


def _mix_rbf_kernel(X, Y, sigmas, wts=None):
    if wts is None:
        wts = [1.0] * sigmas.shape[0]

    # Flatten dimensions similar to tf.tensordot(axes=[[1, 2], [1, 2]])
    X_flat = X.reshape(X.shape[0], -1)
    Y_flat = Y.reshape(Y.shape[0], -1)

    XX = torch.matmul(X_flat, X_flat.t())
    XY = torch.matmul(X_flat, Y_flat.t())
    YY = torch.matmul(Y_flat, Y_flat.t())

    X_sqnorms = torch.diag(XX)
    Y_sqnorms = torch.diag(YY)

    K_XX, K_XY, K_YY = 0., 0., 0.
    for sigma, wt in zip(sigmas, wts):
        gamma = 1 / (2 * sigma ** 2)
        K_XX += wt * torch.exp(-gamma * (-2 * XX + X_sqnorms.unsqueeze(1) + X_sqnorms.unsqueeze(0)))
        K_XY += wt * torch.exp(-gamma * (-2 * XY + X_sqnorms.unsqueeze(1) + Y_sqnorms.unsqueeze(0)))
        K_YY += wt * torch.exp(-gamma * (-2 * YY + Y_sqnorms.unsqueeze(1) + Y_sqnorms.unsqueeze(0)))

    return K_XX, K_XY, K_YY, sum(wts)


def _mmd2(K_XX, K_XY, K_YY, const_diagonal=False, biased=False):
    m = K_XX.shape[0]
    n = K_YY.shape[0]

    if biased:
        mmd2 = (torch.sum(K_XX) / (m * m)
                + torch.sum(K_YY) / (n * n)
                - 2 * torch.sum(K_XY) / (m * n))
    else:
        if const_diagonal is not False:
            trace_X = m * const_diagonal
            trace_Y = n * const_diagonal
        else:
            trace_X = torch.trace(K_XX)
            trace_Y = torch.trace(K_YY)

        mmd2 = ((torch.sum(K_XX) - trace_X) / (m * (m - 1))
                + (torch.sum(K_YY) - trace_Y) / (n * (n - 1))
                - 2 * torch.sum(K_XY) / (m * n))

    return mmd2


def mix_rbf_mmd2(X, Y, sigmas=None, wts=None, biased=True):
    if sigmas is None:
        sigmas = torch.tensor([1.0, 2.0, 4.0, 8.0, 16.0], device=X.device)
    K_XX, K_XY, K_YY, d = _mix_rbf_kernel(X, Y, sigmas, wts)
    return _mmd2(K_XX, K_XY, K_YY, const_diagonal=d, biased=biased)


def max_mean_discrepancy(real_data, syn_data, bandwidths=None):
    if not isinstance(real_data, torch.Tensor):
        X = torch.tensor(real_data, dtype=torch.float32)
    else:
        X = real_data.float()
        
    if not isinstance(syn_data, torch.Tensor):
        Y = torch.tensor(syn_data, dtype=torch.float32)
    else:
        Y = syn_data.float()

    if bandwidths is not None:
        if not isinstance(bandwidths, torch.Tensor):
            bandwidths = torch.tensor(bandwidths, dtype=torch.float32, device=X.device)

    Y = Y.to(X.device)
    mmd2_value = mix_rbf_mmd2(X, Y, sigmas=bandwidths, biased=True) 
    mmd2_value = torch.clamp(mmd2_value, min=0.0) 
    return torch.sqrt(mmd2_value).item()


def pearson_correlation_error(real_data, fake_data):
    """
    Calculates the Mean Absolute Error between the Pearson correlation
    matrices of the real and synthetic features.
    """
    real_flat = real_data.reshape(-1, real_data.shape[2])
    fake_flat = fake_data.reshape(-1, fake_data.shape[2])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        corr_real = np.corrcoef(real_flat, rowvar=False)
        corr_fake = np.corrcoef(fake_flat, rowvar=False)

    corr_real = np.nan_to_num(corr_real, nan=0.0)
    corr_fake = np.nan_to_num(corr_fake, nan=0.0)

    error = np.mean(np.abs(corr_real - corr_fake))
    return error


def evaluate_all(real_c, fake_c, real_d, fake_d):
    print("\n" + "=" * 40)
    print(" EHR-M-GAN EVALUATION METRICS (V2)")
    print("=" * 40)

    # 1. Continuous MMD
    mmd_score = max_mean_discrepancy(real_c, fake_c)
    print(f"Continuous MMD (Lower is better):      {mmd_score:.5f}")

    # 2. Discrete Probability RMSE
    rmse_score = discrete_probability_rmse(real_d, fake_d)
    print(f"Discrete Prob RMSE (Lower is better):  {rmse_score:.5f}")

    # 3. Continuous Feature Correlation Error
    corr_err_c = pearson_correlation_error(real_c, fake_c)
    print(f"Continuous Corr Error (Lower is better): {corr_err_c:.5f}")

    # 4. Discrete Feature Correlation Error
    corr_err_d = pearson_correlation_error(real_d, fake_d)
    print(f"Discrete Corr Error (Lower is better):   {corr_err_d:.5f}")
    print("=" * 40 + "\n")

    return {
        'mmd': float(mmd_score),
        'rmse': float(rmse_score),
        'corr_c': float(corr_err_c),
        'corr_d': float(corr_err_d)
    }


In [ ]:
%%writefile visualise.py
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random
import os
from ultils import renormlizer


def visualise_gan(data_continuous_real, data_continuous_syn, data_discrete_real, data_discrete_syn, inx, max_val_con,
                  min_val_con, num_dim=12, num_plot=10, SAVE_PATH="logs/", c_feature_names=None, d_feature_names=None):
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)

    data_continuous_real = renormlizer(data_continuous_real, max_val_con, min_val_con)
    data_continuous_syn = renormlizer(data_continuous_syn, max_val_con, min_val_con)

    sns.set_style("whitegrid")
    fig, axes = plt.subplots(4, num_dim, figsize=(100, 40))
    fig.suptitle(f"EHR-M-GAN V2 Synthesized Data Validation (Epoch {inx})", fontsize=60, y=0.98)
    plt.setp(axes, xticks=[0, 3, 6, 9, 12, 15, 18, 21, 24])

    c_dim_list = random.sample(list(range(data_continuous_real.shape[2])), num_dim)
    c_pid_index = random.sample(list(range(len(data_continuous_real))), num_plot)
    c_pid_index_syn = random.sample(list(range(len(data_continuous_syn))), num_plot)

    for i in range(num_dim):
        c_title = c_feature_names[c_dim_list[i]] if c_feature_names and c_dim_list[i] < len(c_feature_names) else f"Continuous Feature {c_dim_list[i]}"
        
        # Continuous Real
        ax = axes[0, i]
        df = pd.DataFrame(data_continuous_real[c_pid_index, :, c_dim_list[i]])
        sns.lineplot(ax=ax, data=df.T, palette=sns.color_palette('Greens', n_colors=num_plot), legend=False, alpha=0.3)
        sns.lineplot(ax=ax, data=df.T.mean(axis=1), color='black', linewidth=3, label='Mean')
        ax.set_title(f"Real - {c_title}", fontsize=24)
        ax.set_ylabel("Value", fontsize=20)
        ax.set_xlabel("Time Step (hrs)", fontsize=20)

        # Continuous Synthetic
        ax_syn = axes[1, i]
        df_syn = pd.DataFrame(data_continuous_syn[c_pid_index_syn, :, c_dim_list[i]])
        sns.lineplot(ax=ax_syn, data=df_syn.T, palette=sns.color_palette('Reds', n_colors=num_plot), legend=False, alpha=0.3)
        sns.lineplot(ax=ax_syn, data=df_syn.T.mean(axis=1), color='black', linewidth=3, label='Mean')
        ax_syn.set_title(f"Synthetic - {c_title}", fontsize=24)
        ax_syn.set_ylabel("Value", fontsize=20)
        ax_syn.set_xlabel("Time Step (hrs)", fontsize=20)
        
        # Exact vertical range match with Real plot
        ax_syn.set_ylim(axes[0, i].get_ylim())

    d_dim_list = random.sample(list(range(data_discrete_real.shape[2])), num_dim)
    d_pid_index = random.sample(list(range(len(data_discrete_real))), num_plot)
    d_pid_index_syn = random.sample(list(range(len(data_discrete_syn))), num_plot)

    for i in range(num_dim):
        d_title = d_feature_names[d_dim_list[i]] if d_feature_names and d_dim_list[i] < len(d_feature_names) else f"Discrete Feature {d_dim_list[i]}"
        
        # Discrete Real
        ax = axes[2, i]
        d_data_real = data_discrete_real[d_pid_index, :, d_dim_list[i]]
        if hasattr(d_data_real, "detach"): d_data_real = d_data_real.detach().cpu().numpy()
        
        sns.heatmap(d_data_real, ax=ax, cmap="Greens", cbar=True, vmin=0, vmax=1, linewidths=0.05, linecolor='lightgray')
        ax.set_title(f"Real - {d_title}", fontsize=24)
        ax.set_ylabel("Patient Sample", fontsize=20)
        ax.set_xlabel("Time Step (hrs)", fontsize=20)

        # Discrete Synthetic
        ax = axes[3, i]
        d_data_syn = data_discrete_syn[d_pid_index_syn, :, d_dim_list[i]]
        if hasattr(d_data_syn, "detach"): d_data_syn = d_data_syn.detach().cpu().numpy()

        sns.heatmap(d_data_syn, ax=ax, cmap="Reds", cbar=True, vmin=0, vmax=1, linewidths=0.05, linecolor='lightgray')
        ax.set_title(f"Synthetic - {d_title}", fontsize=24)
        ax.set_ylabel("Patient Sample", fontsize=20)
        ax.set_xlabel("Time Step (hrs)", fontsize=20)

    # Adjust layout to prevent overlap with huge suptitle
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    # Sanitize inx for filename
    safe_inx = str(inx)
    if len(safe_inx) > 100: safe_inx = safe_inx[:100]
    for char in ['{', '}', ':', '"', "'", '[', ']', ' ', '\n', '\t', '\\', '/']:
        safe_inx = safe_inx.replace(char, '_')

    fig.savefig(os.path.join(SAVE_PATH, f'visualise_gan_epoch_{safe_inx}.pdf'), format='pdf')
    plt.close(fig)


def visualise_vae(data_continuous_real, data_continuous_syn, data_discrete_real, data_discrete_syn, inx, max_val_con,
                  min_val_con, num_dim=8, num_plot=10, SAVE_PATH="logs/"):
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)

    data_continuous_real = renormlizer(data_continuous_real, max_val_con, min_val_con)
    data_continuous_syn = renormlizer(data_continuous_syn, max_val_con, min_val_con)

    sns.set_style("whitegrid")
    fig, axes = plt.subplots(2, num_dim, figsize=(100, 30))
    fig.suptitle(f"EHR-M-GAN V2 VAE Reconstruction Validation (Epoch {inx})", fontsize=60, y=0.98)
    plt.setp(axes, xticks=[0, 3, 6, 9, 12, 15, 18, 21, 24])

    c_dim_list = random.sample(list(range(data_continuous_real.shape[2])), num_dim)
    c_pid_index = random.sample(list(range(len(data_continuous_syn))), num_plot)

    for i in range(len(c_dim_list)):
        ax = axes[0, i]
        df = pd.DataFrame(data_continuous_real[c_pid_index, :, c_dim_list[i]])
        sns.lineplot(ax=ax, data=df.T, palette=sns.color_palette('Greens', n_colors=num_plot), legend=False)
        df_syn = pd.DataFrame(data_continuous_syn[c_pid_index, :, c_dim_list[i]])
        sns.lineplot(ax=ax, data=df_syn.T, marker='o', palette=sns.color_palette('Reds', n_colors=num_plot), legend=False)
        ax.set_title(f"Continuous Feature {c_dim_list[i]} (Green=Real, Red=Rec)", fontsize=24)
        ax.set_ylabel("Value", fontsize=20)
        ax.set_xlabel("Time Step (hrs)", fontsize=20)

    d_dim_list = random.sample(list(range(data_discrete_real.shape[2])), num_dim)
    d_pid_index = random.sample(list(range(len(data_discrete_syn))), num_plot)

    for i in range(len(d_dim_list)):
        ax = axes[1, i]
        df = pd.DataFrame(data_discrete_real[d_pid_index, :, d_dim_list[i]])
        sns.lineplot(ax=ax, data=df.T, palette=sns.color_palette('Greens', n_colors=num_plot), legend=False)
        df_syn = pd.DataFrame(data_discrete_syn[d_pid_index, :, d_dim_list[i]])
        sns.lineplot(ax=ax, data=df_syn.T, marker='o', palette=sns.color_palette('Reds', n_colors=num_plot), legend=False)
        ax.set_title(f"Discrete Feature {d_dim_list[i]} (Green=Real, Red=Rec)", fontsize=24)
        ax.set_ylabel("Probability", fontsize=20)
        ax.set_xlabel("Time Step (hrs)", fontsize=20)

    # Adjust layout to prevent overlap with huge suptitle
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    # Sanitize inx for filename
    safe_inx = str(inx)
    if len(safe_inx) > 100: safe_inx = safe_inx[:100]
    for char in ['{', '}', ':', '"', "'", '[', ']', ' ', '\n', '\t', '\\', '/']:
        safe_inx = safe_inx.replace(char, '_')

    fig.savefig(os.path.join(SAVE_PATH, f'visualise_vae_epoch_{safe_inx}.pdf'), format='pdf')
    plt.close(fig)


def plot_metrics_trend(history_dict, save_path="logs/"):
    """
    Plots the quantitative metric trajectory over time and saves as a single multi-panel PDF.
    """
    if not os.path.exists(save_path):
        os.makedirs(save_path)
        
    epochs = history_dict.get('epochs', [])
    if len(epochs) == 0:
        return

    sns.set_style("whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(24, 6))
    fig.suptitle("EHR-M-GAN V2 Quantitative Training Metrics Trend", fontsize=20, y=1.05)

    # Plot 1: MMD
    if 'mmd' in history_dict and len(history_dict['mmd']) > 0:
        axes[0].plot(epochs, history_dict['mmd'], marker='o', color='blue', linewidth=2)
        axes[0].set_title("Continuous MMD (Lower is Better)", fontsize=16)
        axes[0].set_xlabel("Epochs", fontsize=14)
        axes[0].set_ylabel("MMD Score", fontsize=14)
        axes[0].tick_params(axis='both', which='major', labelsize=12)
    
    # Plot 2: RMSE
    if 'rmse' in history_dict and len(history_dict['rmse']) > 0:
        axes[1].plot(epochs, history_dict['rmse'], marker='o', color='green', linewidth=2)
        axes[1].set_title("Discrete Probability RMSE (Lower is Better)", fontsize=16)
        axes[1].set_xlabel("Epochs", fontsize=14)
        axes[1].set_ylabel("RMSE Score", fontsize=14)
        axes[1].tick_params(axis='both', which='major', labelsize=12)
    
    # Plot 3: Correlation Errors
    if 'corr_c' in history_dict and len(history_dict['corr_c']) > 0:
        axes[2].plot(epochs, history_dict['corr_c'], marker='o', label="Continuous Corr Error", color='red', linewidth=2)
        if 'corr_d' in history_dict and len(history_dict['corr_d']) > 0:
            axes[2].plot(epochs, history_dict['corr_d'], marker='s', label="Discrete Corr Error", color='orange', linewidth=2)
        axes[2].set_title("Feature Correlation Error (Lower is Better)", fontsize=16)
        axes[2].set_xlabel("Epochs", fontsize=14)
        axes[2].set_ylabel("Absolute Error", fontsize=14)
        axes[2].legend(fontsize=12)
        axes[2].tick_params(axis='both', which='major', labelsize=12)

    plt.tight_layout()
    fig.savefig(os.path.join(save_path, 'metrics_history.pdf'), format='pdf', bbox_inches='tight')
    plt.close(fig)


In [ ]:
%%writefile trainer.py
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

from networks import AutoregressiveVAE, SequenceDiscriminator, JointGenerator
from ultils import nt_xent_loss, kl_divergence, feature_matching_loss, np_rounding
from visualise import visualise_gan, plot_metrics_trend
from metrics import evaluate_all


def compute_gradient_penalty(discriminator, real_samples, fake_samples, device):
    """Calculates the gradient penalty for WGAN-GP (Enforces Float32 for double-backward stability)."""
    with torch.amp.autocast(device.type, enabled=False):
        alpha = torch.rand(real_samples.size(0), 1, 1, device=device)
        interpolates = (alpha * real_samples.float() + ((1 - alpha) * fake_samples.float())).requires_grad_(True)
        
        # Disable cuDNN benchmarking during gradient penalty calculation to prevent memory leaks
        with torch.backends.cudnn.flags(enabled=False):
            d_interpolates, _ = discriminator(interpolates)
            
        fake = torch.ones_like(d_interpolates)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.reshape(gradients.size(0), -1)
    # Add epsilon to L2 norm calculation to prevent division-by-zero NaN in autograd
    grad_norms = torch.sqrt(torch.sum(gradients ** 2, dim=1) + 1e-12)
    gradient_penalty = ((grad_norms - 1) ** 2).mean()
    return gradient_penalty


def train_m3gan(dataset, config, max_val_con, min_val_con, c_feature_names=None, d_feature_names=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    c_dim, d_dim = config['c_dim'], config['d_dim']
    latent_dim = config['latent_dim']
    hidden_dim = config['hidden_dim']
    noise_dim = config['noise_dim']
    time_steps = config['time_steps']

    # Using VAE Class to wrap the new Autoregressive architecture
    c_vae = AutoregressiveVAE(c_dim, hidden_dim, latent_dim, config['enc_layers'], config['dec_layers'], time_steps).to(device)
    d_vae = AutoregressiveVAE(d_dim, hidden_dim, latent_dim, config['enc_layers'], config['dec_layers'], time_steps).to(device)

    # Initialize Joint Generator (handles bilateral coupling internally, with mapping networks in V2)
    joint_gen = JointGenerator(noise_dim, noise_dim, hidden_dim, latent_dim, latent_dim, config['gen_layers']).to(device)
    c_gen = joint_gen.c_gen
    d_gen = joint_gen.d_gen

    # Spectral Normalization Critic to enforce Lipschitz continuity
    c_dis = SequenceDiscriminator(c_dim, hidden_dim, time_steps, config.get('dis_layers', 3)).to(device)
    d_dis = SequenceDiscriminator(d_dim, hidden_dim, time_steps, config.get('dis_layers', 3)).to(device)

    # Separate parameters for optimizing
    vae_params = list(c_vae.parameters()) + list(d_vae.parameters())
    dis_params = list(c_dis.parameters()) + list(d_dis.parameters())

    optimizer_VAE_pre = optim.Adam(vae_params, lr=config['v_lr_pre'])
    optimizer_VAE = optim.Adam(vae_params, lr=config['v_lr'])
    optimizer_G = optim.Adam(joint_gen.parameters(), lr=config['g_lr'])
    optimizer_D = optim.Adam(dis_params, lr=config['d_lr'])

    scaler = torch.amp.GradScaler('cuda', enabled=config.get('use_amp', False))

    # Resume from checkpoint if specified
    if config.get('resume_checkpoint'):
        print(f"\nLoading checkpoint from {config['resume_checkpoint']}...")
        checkpoint = torch.load(config['resume_checkpoint'], map_location=device)
        
        c_vae.load_state_dict(checkpoint['c_vae'])
        d_vae.load_state_dict(checkpoint['d_vae'])
        print("Successfully loaded VAE weights from checkpoint.")
        
        if not config.get('resume_vae_only', False):
            if 'c_gen' in checkpoint:
                print("Detected pretrained GAN weights in checkpoint. Loading...")
                c_gen.load_state_dict(checkpoint['c_gen'])
                d_gen.load_state_dict(checkpoint['d_gen'])
                c_dis.load_state_dict(checkpoint['c_dis'])
                d_dis.load_state_dict(checkpoint['d_dis'])
        else:
            print("Flag --resume_vae_only is active. Skipping GAN generator/discriminator weight initialization.")

    # ----------------------------------------------------
    # Phase 1: VAE Pretraining (Contrastive VAE Learning)
    # ----------------------------------------------------
    if not config.get('skip_pretrain', False):
        print("\n=== Starting Phase 1: VAE Pretraining ===")
        dataloader = DataLoader(dataset, batch_size=config['pre_batch_size'], shuffle=True, drop_last=True)
        print(f"Phase 1 Batch Size: {config['pre_batch_size']}, Iterations: {len(dataloader)}")

        for epoch in range(config['num_pre_epochs']):
            # Linear KL weight annealing over the first 100 epochs to prevent posterior collapse
            kl_anneal_factor = min(1.0, epoch / 100.0)
            c_real_lst, c_rec_lst, d_real_lst, d_rec_lst = [], [], [], []
            epoch_total_loss = 0.0
            pbar = tqdm(dataloader, desc=f"Pretrain Epoch [{epoch + 1}/{config['num_pre_epochs']}]", leave=True)

            for continuous_x, discrete_x in pbar:
                continuous_x = torch.clamp(torch.nan_to_num(continuous_x.to(device), nan=0.0), 0.0, 1.0)
                discrete_x = torch.clamp(torch.nan_to_num(discrete_x.to(device), nan=0.0), 0.0, 1.0)

                optimizer_VAE_pre.zero_grad()

                with torch.amp.autocast(device.type, enabled=config.get('use_amp', False)):
                    c_rec, _, c_mu, c_logvar, c_z = c_vae(continuous_x)
                    d_rec, d_logits, d_mu, d_logvar, d_z = d_vae(discrete_x)

                    loss_c_rec = F.mse_loss(c_rec, continuous_x)
                    loss_d_rec = F.binary_cross_entropy_with_logits(d_logits, discrete_x)

                    loss_c_kl = kl_divergence(c_mu, c_logvar)
                    loss_d_kl = kl_divergence(d_mu, d_logvar)

                    c_z_flat = c_z.view(c_z.size(0), -1)
                    d_z_flat = d_z.view(d_z.size(0), -1)
                    loss_contrastive = nt_xent_loss(c_z_flat, d_z_flat)
                    loss_matching = F.mse_loss(c_z, d_z)

                    total_vae_loss = (config['alpha_re'] * (loss_c_rec + loss_d_rec) +
                                      (config['alpha_kl'] * kl_anneal_factor) * (loss_c_kl + loss_d_kl) +
                                      config['alpha_ct'] * loss_contrastive +
                                      config['alpha_mt'] * loss_matching)

                scaler.scale(total_vae_loss).backward()
                # Unscale gradients before clipping when using AMP
                scaler.unscale_(optimizer_VAE_pre)
                torch.nn.utils.clip_grad_norm_(vae_params, max_norm=5.0)
                scaler.step(optimizer_VAE_pre)
                scaler.update()

                epoch_total_loss += total_vae_loss.item()
                pbar.set_postfix(
                    c_rec=f"{loss_c_rec.item():.4f}",
                    d_rec=f"{loss_d_rec.item():.4f}",
                    tot_vae=f"{total_vae_loss.item():.4f}"
                )

            print(f"Pretrain Epoch {epoch + 1}: Loss = {epoch_total_loss / len(dataloader):.4f}")
            
            # Save periodic pretrain check
            if (epoch + 1) % 50 == 0:
                os.makedirs("Output/checkpoint", exist_ok=True)
                torch.save({
                    'c_vae': c_vae.state_dict(),
                    'd_vae': d_vae.state_dict()
                }, f"Output/checkpoint/vae_pretrain_{epoch + 1}.pth")

    # ----------------------------------------------------
    # Phase 2: Joint VAE-GAN Training
    # ----------------------------------------------------
    print("\n=== Starting Phase 2: Joint VAE-GAN Training (V2) ===")
    dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True, drop_last=True)
    
    # Trackers for quantitative validation
    history_dict = {'epochs': [], 'mmd': [], 'rmse': [], 'corr_c': [], 'corr_d': []}
    best_gan_score = float('inf')

    for epoch in range(config['num_epochs']):
        c_dis_loss_lst, d_dis_loss_lst, g_loss_lst, vae_loss_lst = [], [], [], []
        pbar = tqdm(dataloader, desc=f"GAN Epoch [{epoch + 1}/{config['num_epochs']}]", leave=True)

        for continuous_x, discrete_x in pbar:
            continuous_x = torch.clamp(torch.nan_to_num(continuous_x.to(device), nan=0.0), 0.0, 1.0)
            discrete_x = torch.clamp(torch.nan_to_num(discrete_x.to(device), nan=0.0), 0.0, 1.0)
            batch_size = continuous_x.size(0)

            # ----------------------------------------------------
            # 1. TRAIN DISCRIMINATORS (CRITICS)
            # ----------------------------------------------------
            for _ in range(config['d_rounds']):
                optimizer_D.zero_grad()

                # Generate fake latents
                noise_c = torch.randn(batch_size, time_steps, noise_dim, device=device)
                noise_d = torch.randn(batch_size, time_steps, noise_dim, device=device)

                with torch.no_grad():
                    fake_z_c, fake_z_d = joint_gen(noise_c, noise_d)
                    fake_c_seq, _ = c_vae.reconstruct_decoder(fake_z_c)
                    fake_d_seq, _ = d_vae.reconstruct_decoder(fake_z_d)

                with torch.amp.autocast(device.type, enabled=config.get('use_amp', False)):
                    # Compute Wasserstein Loss with spectral norm and gradient penalty
                    real_c_score, _ = c_dis(continuous_x)
                    fake_c_score, _ = c_dis(fake_c_seq.detach())

                    real_d_score, _ = d_dis(discrete_x)
                    fake_d_score, _ = d_dis(fake_d_seq.detach())

                    loss_c_gp = compute_gradient_penalty(c_dis, continuous_x, fake_c_seq.detach(), device)
                    loss_d_gp = compute_gradient_penalty(d_dis, discrete_x, fake_d_seq.detach(), device)

                    # Wasserstein Critics minimize Wasserstein distance negative
                    loss_c_dis = fake_c_score.mean() - real_c_score.mean() + 10 * loss_c_gp
                    loss_d_dis = fake_d_score.mean() - real_d_score.mean() + 10 * loss_d_gp
                    total_dis_loss = config.get('c_beta_adv', 1.0) * loss_c_dis + config.get('d_beta_adv', 1.0) * loss_d_dis

                scaler.scale(total_dis_loss).backward()
                # Unscale gradients before clipping when using AMP
                scaler.unscale_(optimizer_D)
                torch.nn.utils.clip_grad_norm_(dis_params, max_norm=5.0)
                scaler.step(optimizer_D)
                scaler.update()

                c_dis_loss_lst.append(loss_c_dis.item())
                d_dis_loss_lst.append(loss_d_dis.item())

            # ----------------------------------------------------
            # 2. TRAIN GENERATOR
            # ----------------------------------------------------
            optimizer_G.zero_grad()

            with torch.amp.autocast(device.type, enabled=config.get('use_amp', False)):
                noise_c = torch.randn(batch_size, time_steps, noise_dim, device=device)
                noise_d = torch.randn(batch_size, time_steps, noise_dim, device=device)

                # Bilateral Coupled Forward pass
                fake_z_c, fake_z_d = joint_gen(noise_c, noise_d)

                fake_c_seq, _ = c_vae.reconstruct_decoder(fake_z_c)
                fake_d_seq, _ = d_vae.reconstruct_decoder(fake_z_d)

                # Extract features from critics to apply Feature Matching loss
                fake_c_score, fake_c_feat = c_dis(fake_c_seq)
                fake_d_score, fake_d_feat = d_dis(fake_d_seq)

                with torch.no_grad():
                    _, real_c_feat = c_dis(continuous_x)
                    _, real_d_feat = d_dis(discrete_x)

                loss_c_gen = -fake_c_score.mean()
                loss_d_gen = -fake_d_score.mean()

                loss_c_fm = feature_matching_loss(fake_c_feat, real_c_feat)
                loss_d_fm = feature_matching_loss(fake_d_feat, real_d_feat)

                # Generator loss consists of adversarial and feature matching metrics
                total_gen_loss = (config.get('c_beta_adv', 1.0) * loss_c_gen +
                                  config.get('d_beta_adv', 1.0) * loss_d_gen +
                                  config.get('c_beta_fm', 10.0) * loss_c_fm +
                                  config.get('d_beta_fm', 10.0) * loss_d_fm)

            scaler.scale(total_gen_loss).backward()
            # Unscale gradients before clipping when using AMP
            scaler.unscale_(optimizer_G)
            torch.nn.utils.clip_grad_norm_(joint_gen.parameters(), max_norm=5.0)
            scaler.step(optimizer_G)
            scaler.update()
            g_loss_lst.append(total_gen_loss.item())

            # ----------------------------------------------------
            # 3. TRAIN VAE (Joint VAE Loss)
            # ----------------------------------------------------
            optimizer_VAE.zero_grad()

            with torch.amp.autocast(device.type, enabled=config.get('use_amp', False)):
                c_rec, _, c_mu, c_logvar, c_z = c_vae(continuous_x)
                d_rec, d_logits, d_mu, d_logvar, d_z = d_vae(discrete_x)

                loss_c_rec = F.mse_loss(c_rec, continuous_x)
                loss_d_rec = F.binary_cross_entropy_with_logits(d_logits, discrete_x)

                loss_c_kl = kl_divergence(c_mu, c_logvar)
                loss_d_kl = kl_divergence(d_mu, d_logvar)

                c_z_flat = c_z.view(c_z.size(0), -1)
                d_z_flat = d_z.view(d_z.size(0), -1)
                loss_contrastive = nt_xent_loss(c_z_flat, d_z_flat)
                loss_matching = F.mse_loss(c_z, d_z)

                total_vae_loss = (config['alpha_re'] * (loss_c_rec + loss_d_rec) +
                                  config['alpha_kl'] * (loss_c_kl + loss_d_kl) +
                                  config['alpha_ct'] * loss_contrastive +
                                  config['alpha_mt'] * loss_matching)

            scaler.scale(total_vae_loss).backward()
            # Unscale gradients before clipping when using AMP
            scaler.unscale_(optimizer_VAE)
            torch.nn.utils.clip_grad_norm_(vae_params, max_norm=5.0)
            scaler.step(optimizer_VAE)
            scaler.update()
            vae_loss_lst.append(total_vae_loss.item())

            pbar.set_postfix(
                D_c=f"{np.mean(c_dis_loss_lst):.3f}",
                D_d=f"{np.mean(d_dis_loss_lst):.3f}",
                G=f"{np.mean(g_loss_lst):.3f}",
                VAE=f"{np.mean(vae_loss_lst):.3f}"
            )

        # ----------------------------------------------------
        # Epoch-based Validation & Fast Metric Calculation
        # ----------------------------------------------------
        if (epoch + 1) % 5 == 0:
            print(f"\n✅ Epoch {epoch + 1} completed.")

        if (epoch + 1) % config.get('epoch_ckpt_freq', 100) == 0 or (epoch + 1) == config['num_epochs']:
            # Run Fast Metric Evaluation on 1000 generated sequences
            eval_batches = 1000 // config['batch_size']
            if eval_batches == 0: eval_batches = 1
            
            c_real_eval, d_real_eval = [], []
            c_gen_eval, d_gen_eval = [], []

            c_vae.eval()
            d_vae.eval()
            joint_gen.eval()

            # Grab some real comparison batches
            for idx, (real_c, real_d) in enumerate(dataloader):
                if idx >= eval_batches:
                    break
                c_real_eval.append(real_c.numpy())
                d_real_eval.append(real_d.numpy())

            with torch.no_grad():
                for _ in range(eval_batches):
                    noise_c = torch.randn(config['batch_size'], time_steps, noise_dim, device=device)
                    noise_d = torch.randn(config['batch_size'], time_steps, noise_dim, device=device)

                    # ROOT FIX: Huấn luyện dùng joint_gen thì đánh giá cũng PHẢI dùng joint_gen!
                    # Không dùng c_gen/d_gen riêng lẻ vì chúng bị thiếu Attention và bị ép qua Sigmoid.
                    fake_z_c, fake_z_d = joint_gen(noise_c, noise_d)

                    fake_c_seq, _ = c_vae.reconstruct_decoder(fake_z_c)
                    fake_d_seq, _ = d_vae.reconstruct_decoder(fake_z_d)

                    c_gen_eval.append(fake_c_seq.cpu().numpy())
                    d_gen_eval.append(fake_d_seq.cpu().numpy())

            c_real_eval = np.concatenate(c_real_eval, axis=0)
            d_real_eval = np.concatenate(d_real_eval, axis=0)
            c_gen_eval = np.concatenate(c_gen_eval, axis=0)
            d_gen_eval = np.concatenate(d_gen_eval, axis=0)

            # Round discrete probabilities
            d_gen_rounded = np_rounding(d_gen_eval)

            # Evaluate Metrics
            scores = evaluate_all(c_real_eval, c_gen_eval, d_real_eval, d_gen_rounded)
            
            # Save history
            history_dict['epochs'].append(epoch + 1)
            history_dict['mmd'].append(scores['mmd'])
            history_dict['rmse'].append(scores['rmse'])
            history_dict['corr_c'].append(scores['corr_c'])
            history_dict['corr_d'].append(scores['corr_d'])

            # Plot training metrics trend and patient curves
            plot_metrics_trend(history_dict, save_path="Output/test_plots/")
            visualise_gan(
                c_real_eval, c_gen_eval, d_real_eval, d_gen_eval,
                epoch + 1, max_val_con, min_val_con,
                SAVE_PATH="Output/test_plots/",
                c_feature_names=c_feature_names,
                d_feature_names=d_feature_names
            )

            # Restore models back to training mode
            c_vae.train()
            d_vae.train()
            joint_gen.train()

            # Save checkpoint
            os.makedirs("Output/checkpoint", exist_ok=True)
            checkpoint_path = f"Output/checkpoint/m3gan_v2_epoch_{epoch + 1}.pth"
            torch.save({
                'epoch': epoch + 1,
                'c_gen': c_gen.state_dict(),
                'd_gen': d_gen.state_dict(),
                'c_vae': c_vae.state_dict(),
                'd_vae': d_vae.state_dict(),
                'c_dis': c_dis.state_dict(),
                'd_dis': d_dis.state_dict(),
                'history': history_dict
            }, checkpoint_path)
            print(f"Saved checkpoint to {checkpoint_path}")

            # Keep track of the best model (MMD + RMSE score combined)
            combined_score = scores['mmd'] + scores['rmse']
            if combined_score < best_gan_score:
                best_gan_score = combined_score
                best_checkpoint_path = "Output/checkpoint/best_m3gan_v2.pth"
                torch.save({
                    'epoch': epoch + 1,
                    'c_gen': c_gen.state_dict(),
                    'd_gen': d_gen.state_dict(),
                    'c_vae': c_vae.state_dict(),
                    'd_vae': d_vae.state_dict(),
                    'c_dis': c_dis.state_dict(),
                    'd_dis': d_dis.state_dict(),
                    'scores': scores
                }, best_checkpoint_path)
                print(f"🥇 New best model found! Saved to {best_checkpoint_path}")


In [ ]:
%%writefile main.py
import torch
import os
import numpy as np
import pickle
import argparse
from torch.utils.data import TensorDataset

from trainer import train_m3gan


def main(args):
    # Setup dataset paths
    data_path = os.path.join('Data/', args.dataset)
    
    vital_path = os.path.join(data_path, 'vital_sign_24hrs.pkl')
    med_path = os.path.join(data_path, 'med_interv_24hrs.pkl')
    
    if not os.path.exists(vital_path) or not os.path.exists(med_path):
        raise FileNotFoundError(f"Dataset not found at {data_path}. Please make sure 'vital_sign_24hrs.pkl' and 'med_interv_24hrs.pkl' exist.")

    # Load vital signs (continuous) and medication interventions (discrete)
    with open(vital_path, 'rb') as f:
        continuous_x = pickle.load(f)
    with open(med_path, 'rb') as f:
        discrete_x = pickle.load(f)

    # Preprocess discrete and continuous features
    discrete_x = np.nan_to_num(np.clip(discrete_x, 0.0, 1.0), nan=0.0)
    continuous_x = np.nan_to_num(continuous_x, nan=0.0)

    # Normalize continuous vital signs to [0, 1] range
    min_val_con = np.min(continuous_x, axis=(0, 1))
    max_val_con = np.max(continuous_x, axis=(0, 1))
    range_con = max_val_con - min_val_con
    range_con[range_con == 0.0] = 1e-6 # Avoid division by zero
    
    continuous_x = (continuous_x - min_val_con) / range_con

    # Prepare configuration dictionary
    config = {
        'batch_size': args.batch_size,
        'pre_batch_size': args.pre_batch_size,
        'time_steps': continuous_x.shape[1],
        'c_dim': continuous_x.shape[2],
        'd_dim': discrete_x.shape[2],
        'latent_dim': 25,
        'hidden_dim': args.gen_num_units,
        'noise_dim': min(continuous_x.shape[2] // 2, discrete_x.shape[2] // 2),
        'enc_layers': args.enc_layers,
        'dec_layers': args.dec_layers,
        'gen_layers': args.gen_num_layers,
        'dis_layers': args.dis_num_layers,
        'num_pre_epochs': args.num_pre_epochs,
        'num_epochs': args.num_epochs,
        'epoch_ckpt_freq': args.epoch_ckpt_freq,
        'v_lr_pre': args.v_lr_pre,
        'v_lr': args.v_lr,
        'g_lr': args.g_lr,
        'd_lr': args.d_lr,
        'd_rounds': args.d_rounds,
        'g_rounds': args.g_rounds,
        'v_rounds': args.v_rounds,
        'alpha_re': args.alpha_re,
        'alpha_kl': args.alpha_kl,
        'alpha_mt': args.alpha_mt,
        'alpha_ct': args.alpha_ct,
        'c_beta_adv': args.c_beta_adv,
        'c_beta_fm': args.c_beta_fm,
        'd_beta_adv': args.d_beta_adv,
        'd_beta_fm': args.d_beta_fm,
        'skip_pretrain': args.skip_pretrain,
        'use_amp': args.use_amp,
        'resume_checkpoint': args.resume_checkpoint,
        'resume_vae_only': args.resume_vae_only
    }

    # Load custom scale names if available
    c_feature_names = None
    d_feature_names = None
    
    # Feature columns names (for MIMIC-III standard)
    if args.dataset.lower() == 'mimic':
        c_feature_names = ['Heart Rate', 'Systolic BP', 'Diastolic BP', 'Mean BP', 'Resp Rate', 'Temp C', 'SpO2', 'Glucose']
        d_feature_names = ['Adenosine', 'Dobutamine', 'Dopamine', 'Epinephrine', 'Isuprel', 'Milrinone', 'Norepinephrine', 'Phenylephrine', 'Vasopressin']

    # Convert to Tensor Dataset and trigger training
    dataset = TensorDataset(
        torch.tensor(continuous_x, dtype=torch.float32),
        torch.tensor(discrete_x, dtype=torch.float32)
    )

    print("\n" + "=" * 50)
    print(" EHR-M-GAN V2 TRAINING INITIATED")
    print(f" Dataset:    {args.dataset}")
    print(f" Devices:    {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")
    print(f" Pre-Epochs: {args.num_pre_epochs} (Skip: {args.skip_pretrain})")
    print(f" GAN Epochs: {args.num_epochs}")
    print(f" TTUR:       d_lr={args.d_lr}, g_lr={args.g_lr}")
    print(f" WGAN-GP:    d_rounds={args.d_rounds}")
    print("=" * 50 + "\n")

    train_m3gan(dataset, config, range_con, min_val_con, c_feature_names, d_feature_names)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description="EHR-M-GAN V2 Command Line Interface")
    parser.add_argument('--dataset', type=str, default="mimic", help="Dataset directory under 'Data/' (e.g., mimic)")
    parser.add_argument('--batch_size', type=int, default=256, help="Batch size for Phase 2 joint training")
    parser.add_argument('--pre_batch_size', type=int, default=1024, help="Batch size for Phase 1 VAE pretraining")
    parser.add_argument('--num_pre_epochs', type=int, default=500, help="Number of pretraining epochs")
    parser.add_argument('--num_epochs', type=int, default=800, help="Number of joint VAE-GAN training epochs")
    parser.add_argument('--epoch_ckpt_freq', type=int, default=50, help="Epoch frequency for evaluation and checkpoints")
    
    # Optimizers learning rates (TTUR settings optimized for WGAN-GP by default)
    parser.add_argument('--v_lr_pre', type=float, default=0.0005, help="Learning rate for Phase 1 VAE")
    parser.add_argument('--v_lr', type=float, default=0.0001, help="Learning rate for Phase 2 VAE")
    parser.add_argument('--g_lr', type=float, default=0.0001, help="Learning rate for Phase 2 Generator")
    parser.add_argument('--d_lr', type=float, default=0.0003, help="Learning rate for Phase 2 Discriminator (TTUR)")
    
    # Critic updates per generator update (standard WGAN-GP default = 5)
    parser.add_argument('--d_rounds', type=int, default=5, help="Number of discriminator updates per G step")
    parser.add_argument('--g_rounds', type=int, default=1)
    parser.add_argument('--v_rounds', type=int, default=1)
    
    # VAE Loss Scaling weights
    parser.add_argument('--alpha_re', type=float, default=1.0, help="Reconstruction loss weight")
    parser.add_argument('--alpha_kl', type=float, default=0.5, help="KL divergence loss weight")
    parser.add_argument('--alpha_mt', type=float, default=0.1, help="Bilateral latent matching loss weight")
    parser.add_argument('--alpha_ct', type=float, default=0.1, help="Latent contrastive loss weight")
    
    # GAN Loss Scaling weights
    parser.add_argument('--c_beta_adv', type=float, default=1.0, help="Continuous GAN loss scaling")
    parser.add_argument('--c_beta_fm', type=float, default=10.0, help="Continuous Feature Matching loss scaling")
    parser.add_argument('--d_beta_adv', type=float, default=1.0, help="Discrete GAN loss scaling")
    parser.add_argument('--d_beta_fm', type=float, default=10.0, help="Discrete Feature Matching loss scaling")
    
    # Network layers
    parser.add_argument('--enc_layers', type=int, default=3, help="Number of layers in VAE Encoder LSTM")
    parser.add_argument('--dec_layers', type=int, default=3, help="Number of layers in VAE Decoder LSTM")
    parser.add_argument('--gen_num_units', type=int, default=512, help="Hidden units in Bilateral cells")
    parser.add_argument('--gen_num_layers', type=int, default=3, help="Number of layers in Generator LSTMs")
    parser.add_argument('--dis_num_layers', type=int, default=3, help="Number of layers in Discriminator LSTMs")
    
    parser.add_argument('--skip_pretrain', action='store_true', default=False, help="Skip Phase 1 and go straight to joint VAE-GAN training")
    parser.add_argument('--use_amp', action='store_true', default=True, help="Enable PyTorch mixed precision (AMP) for speed")
    parser.add_argument('--resume_checkpoint', type=str, default=None, help="Path to checkpoint .pth file to resume training")
    parser.add_argument('--resume_vae_only', action='store_true', default=False, help="If true, only load VAE weights from checkpoint and skip GAN weights.")

    main(parser.parse_args())


In [ ]:
# Start Training!
# Update --resume_checkpoint with your actual path if resuming
!python main.py --dataset mimic --skip_pretrain --use_amp --batch_size 128 --epoch_ckpt_freq 10 --num_epochs 150